# B6: CleanLab + Baseline Summary

**Leaf-JEPA IRP** | Stage 3 — Baseline Establishment

## Rationale

B6 has two jobs:
1. **CleanLab**: Detect likely mislabelled samples using B1's out-of-fold probabilities
2. **Summary**: Aggregate all baseline results into the dissertation comparison table and figures

This notebook works partially (B1-B4 only) and fully (B1-B5). Run it after B1-B4 for
early analysis, then re-run after B5 for the complete picture.

## Initialization

In [3]:
%pip install cleanlab

Note: you may need to restart the kernel to use updated packages.Collecting cleanlab
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
Using cached termcolor-3.3.0-py3-none-any.whl (7.7 kB)

   -------------------- ------------------- 1/2 [cleanlab]
   -------------------- ------------------- 1/2 [cleanlab]
   -------------------- ------------------- 1/2 [cleanlab]
   -------------------- ------------------- 1/2 [cleanlab]
   -------------------- ------------------- 1/2 [cleanlab]
   -------------------- ------------------- 1/2 [cleanlab]
   -------------------- ------------------- 1/2 [cleanlab]
   -------------------- ------------------- 1/2 [cleanlab]
   -------------------- ------------------- 1/2 [cleanlab]
   ---------------------------------------- 2/2 [cleanlab]





[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path("../..").resolve()))
sys.path.insert(0, str(Path("..").resolve()))

from stage3_baseline_establishment.config_stage3 import *
from stage3_baseline_establishment.baseline_utils import save_results, load_split, seed_everything

seed_everything(RANDOM_SEED)
BASELINES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

_, _, class_names = load_split(SPLITS_DIR / "train_split.json")
print(f"Classes: {len(class_names)}")

Classes: 38


## CleanLab label noise detection

In [2]:
oof_probs_path = BASELINES_DIR / "B1_oof_probabilities.npy"
oof_labels_path = BASELINES_DIR / "B1_oof_labels.npy"

if oof_probs_path.exists() and oof_labels_path.exists():
    print("Running CleanLab label noise detection...")
    oof_probs = np.load(oof_probs_path)
    oof_labels = np.load(oof_labels_path)
    print(f"  OOF probs shape: {oof_probs.shape}")
    print(f"  OOF labels shape: {oof_labels.shape}")

    try:
        from cleanlab.filter import find_label_issues
        from cleanlab.rank import get_label_quality_scores

        quality_scores = get_label_quality_scores(oof_labels, oof_probs)
        issue_mask = find_label_issues(oof_labels, oof_probs)

        n_issues = issue_mask.sum()
        severe_mask = quality_scores < CLEANLAB_QUALITY_THRESHOLD
        n_severe = severe_mask.sum()

        print(f"\n  Total label issues found: {n_issues}")
        print(f"  Severe (quality < {CLEANLAB_QUALITY_THRESHOLD}): {n_severe}")

        # Per-class breakdown
        issue_classes = oof_labels[issue_mask]
        class_issue_counts = {}
        for c in range(NUM_CLASSES):
            count = (issue_classes == c).sum()
            if count > 0:
                name = class_names[c] if c < len(class_names) else str(c)
                class_issue_counts[name] = int(count)

        # Sort by count
        sorted_issues = dict(sorted(class_issue_counts.items(),
                                     key=lambda x: x[1], reverse=True))

        print(f"\n  Top classes with label issues:")
        for name, count in list(sorted_issues.items())[:10]:
            print(f"    {name}: {count}")

        cleanlab_results = {
            "total_samples": len(oof_labels),
            "total_issues": int(n_issues),
            "severe_issues": int(n_severe),
            "quality_threshold": CLEANLAB_QUALITY_THRESHOLD,
            "per_class_issues": sorted_issues,
            "mean_quality_score": float(quality_scores.mean()),
            "median_quality_score": float(np.median(quality_scores)),
        }
        save_results(cleanlab_results, BASELINES_DIR / "B6_cleanlab_results.json")

    except ImportError:
        print("  cleanlab not installed. Run: pip install cleanlab")
        print("  Skipping CleanLab analysis.")
else:
    print("  OOF probabilities not found. Run B1 first (with 5-fold CV cell).")
    print(f"  Expected: {oof_probs_path}")

Running CleanLab label noise detection...
  OOF probs shape: (38013, 38)
  OOF labels shape: (38013,)

  Total label issues found: 112
  Severe (quality < 0.1): 207

  Top classes with label issues:
    Corn_(maize)___Northern_Leaf_Blight: 41
    Tomato___Spider_mites Two-spotted_spider_mite: 20
    Tomato___Late_blight: 14
    Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 10
    Tomato___Early_blight: 9
    Corn_(maize)___Common_rust_: 7
    Potato___Late_blight: 5
    Tomato___Septoria_leaf_spot: 3
    Tomato___Target_Spot: 2
    Grape___Black_rot: 1
  Saved: D:\IRP\leaf-jepa\stage3_baseline_establishment\outputs\baselines\B6_cleanlab_results.json


## Aggregate baseline results

In [3]:
print("\n" + "=" * 60)
print("  BASELINE SUMMARY")
print("=" * 60)

baseline_ids = ["B1", "B2", "B3", "B4", "B5"]
rows = []

for bid in baseline_ids:
    agg_path = BASELINES_DIR / f"{bid}_aggregate.json"
    if agg_path.exists():
        with open(agg_path) as f:
            agg = json.load(f)
        rows.append({
            "Baseline": bid,
            "Model": agg.get("model", ""),
            "Macro-F1": agg["macro_f1"],
            "Accuracy": agg["accuracy"],
            "Trainable Params": agg.get("trainable_params", ""),
            "Training Time (s)": agg.get("training_time_s", ""),
            "Best Val F1": agg.get("best_val_f1", ""),
        })
        print(f"  {bid}: F1={agg['macro_f1']:.4f}  Acc={agg['accuracy']:.4f}")
    else:
        print(f"  {bid}: NOT FOUND (run notebook first)")

if rows:
    df = pd.DataFrame(rows)
    csv_path = BASELINES_DIR / "baseline_summary_table.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n  Summary table saved: {csv_path}")
    print(df.to_string(index=False))


  BASELINE SUMMARY
  B1: F1=0.9853  Acc=0.9896
  B2: F1=0.9103  Acc=0.9379
  B3: NOT FOUND (run notebook first)
  B4: NOT FOUND (run notebook first)
  B5: NOT FOUND (run notebook first)

  Summary table saved: D:\IRP\leaf-jepa\stage3_baseline_establishment\outputs\baselines\baseline_summary_table.csv
Baseline           Model  Macro-F1  Accuracy  Trainable Params  Training Time (s)  Best Val F1
      B1       ResNet-50  0.985307  0.989565          23585894         298.395787     0.990371
      B2 EfficientNet-B3  0.910260  0.937884          10754638         366.646808     0.910592


## Label efficiency comparison plot

In [4]:
print("\nGenerating label efficiency comparison plot...")

fig, ax = plt.subplots(figsize=(10, 6))
colours = {"B1": "#e74c3c", "B2": "#3498db"}
markers = {"B1": "o", "B2": "s"}

for bid in ["B1", "B2"]:
    le_path = BASELINES_DIR / f"{bid}_label_efficiency.json"
    if not le_path.exists():
        print(f"  {bid} label efficiency results not found, skipping.")
        continue

    with open(le_path) as f:
        le_data = json.load(f)

    # Group by fraction, compute mean +/- std across seeds
    frac_results = {}
    for key, val in le_data.items():
        frac = val["fraction"]
        if frac not in frac_results:
            frac_results[frac] = []
        frac_results[frac].append(val["macro_f1"])

    fracs = sorted(frac_results.keys())
    means = [np.mean(frac_results[f]) for f in fracs]
    stds  = [np.std(frac_results[f]) for f in fracs]

    ax.errorbar(fracs, means, yerr=stds, marker=markers[bid], color=colours[bid],
                label=bid, capsize=4, linewidth=2, markersize=8)

ax.set_xlabel("Label Fraction", fontsize=12)
ax.set_ylabel("Macro-F1", fontsize=12)
ax.set_title("Label Efficiency: CNN Baselines", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.05)
ax.set_ylim(0, 1.05)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "baseline_label_efficiency_comparison.png",
            dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Label efficiency plot saved.")


Generating label efficiency comparison plot...
  B2 label efficiency results not found, skipping.
  Label efficiency plot saved.


## Per-class F1 heatmap

In [5]:
print("\nGenerating per-class F1 heatmap...")

heatmap_data = {}
for bid in baseline_ids:
    agg_path = BASELINES_DIR / f"{bid}_aggregate.json"
    if agg_path.exists():
        with open(agg_path) as f:
            agg = json.load(f)
        if "per_class_f1" in agg:
            heatmap_data[bid] = agg["per_class_f1"]

if heatmap_data:
    df_hm = pd.DataFrame(heatmap_data, index=class_names[:len(list(heatmap_data.values())[0])])
    fig, ax = plt.subplots(figsize=(max(8, len(heatmap_data) * 2), 16))
    sns.heatmap(df_hm, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
                ax=ax, linewidths=0.5, annot_kws={"fontsize": 7})
    ax.set_title("Per-Class F1 Across Baselines", fontsize=14)
    ax.set_ylabel("")
    plt.yticks(fontsize=7)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "B6_per_class_f1_heatmap.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  Per-class F1 heatmap saved.")

    # Save as CSV too
    df_hm.to_csv(BASELINES_DIR / "B6_per_class_f1_all_baselines.csv")
else:
    print("  No per-class F1 data available yet.")

print("\nB6 complete.")


Generating per-class F1 heatmap...
  Per-class F1 heatmap saved.

B6 complete.


## Critical Analysis

B6 closes Stage 3 with two key contributions:

1. **CleanLab findings** — Document how many samples were flagged, which classes were most affected, and whether the flagged pairs match Stage 2's inter-class similarity predictions. Even zero issues found is a positive finding (data quality confirmed).

2. **Summary table** — This is the first table in your Results chapter. It should show all baselines with macro-F1, accuracy, trainable parameters, and training time. The narrative flows: supervised CNNs set the floor (B1, B2), generic SSL provides decent representations (B3), full fine-tuning establishes the ceiling (B4), and domain pretraining improves representation quality (B5 > B3).

3. **Label efficiency curves** — B1 and B2 curves establish how quickly supervised models degrade with fewer labels. Stage 5 PEFT methods will be plotted on the same axes.

4. **Per-class heatmap** — Identifies which diseases are universally hard vs method-dependent. This feeds into Stage 6 error analysis.